# Виявлення шахрайських користувачів у платіжних транзакціях

## Мета роботи
Мета цього дослідження — побудувати модель для класифікації користувачів за ознакою `is_fraud` на основі інформації про користувачів та історії їхніх транзакцій.

У роботі виконується:
- первинний аналіз даних;
- обробка пропусків і перетворення часових полів;
- побудова агрегованих ознак на рівні користувача;
- навчання baseline-моделей;
- покращення якості моделі за рахунок нових фіч;
- підбір оптимального порогу класифікації для підвищення F1-score.

Основна метрика якості — **F1-score**, оскільки задача має суттєвий дисбаланс класів.

## 1. Імпорт бібліотек і завантаження даних

На цьому етапі підключаємо необхідні бібліотеки та завантажуємо чотири основні файли:
- `train_transactions.csv`
- `train_users.csv`
- `test_transactions.csv`
- `test_users.csv`

Транзакційні таблиці містять історію платежів, а таблиці користувачів — базові атрибути акаунтів. Цільова змінна `is_fraud` наявна лише в `train_users`.

In [1]:
import pandas as pd
import numpy as np

train_transactions = pd.read_csv("train_transactions.csv")
train_users = pd.read_csv("train_users.csv")

test_transactions = pd.read_csv("test_transactions.csv")
test_users = pd.read_csv("test_users.csv")

## 2. Розмір датасетів

Перевіримо розміри всіх таблиць. Це допомагає зрозуміти масштаб задачі та співвідношення між кількістю користувачів і кількістю транзакцій.

Очікувано, кількість транзакцій значно більша за кількість користувачів, тому надалі транзакції потрібно буде агрегувати до рівня користувача.

In [2]:
print(train_transactions.shape)
print(train_users.shape)

print(test_transactions.shape)
print(test_users.shape)

(3135378, 13)
(395381, 7)
(1353503, 13)
(169449, 6)


## 3. Первинний огляд даних

Подивимося на перші рядки таблиць, щоб зрозуміти структуру даних, назви колонок і приклади значень.

Транзакційна таблиця містить часові мітки, суму, статус, тип транзакції, географічні та карткові атрибути.  
Таблиця користувачів містить дату реєстрації, email, країну реєстрації, тип трафіку та цільову змінну `is_fraud`.

In [3]:
print(train_transactions.head())
print()
print(train_users.head())

    id_user                      timestamp_tr  amount   status  \
0  15383249         2025-09-06 07:45:39+00:00    3.81     fail   
1   9458117         2025-10-10 11:23:30+00:00    7.94  success   
2  21312302         2025-09-07 19:48:45+00:00    3.81  success   
3     61828  2025-01-02 10:25:02.150802+00:00    3.48     fail   
4  13164211         2025-12-23 21:56:32+00:00   29.25  success   

  transaction_type error_group currency  card_brand card_type card_country  \
0   card_recurring   antifraud      EUR  MASTERCARD     DEBIT       Sweden   
1        card_init         NaN      EUR        VISA     DEBIT      Romania   
2   card_recurring         NaN      EUR  MASTERCARD     DEBIT      Austria   
3        card_init   antifraud      EUR        VISA     DEBIT     Portugal   
4   card_recurring         NaN      EUR  MASTERCARD    CREDIT      Belgium   

      card_holder    card_mask_hash payment_country  
0  johansson lars  24fe124163d8a8fd          Sweden  
1      mark evans  c363e0d

## 4. Перевірка типів даних

Перевіряємо типи колонок, щоб побачити, які змінні є числовими, які — текстовими, а які потребують додаткового перетворення.

Особливу увагу звертаємо на часові колонки:
- `timestamp_tr`
- `timestamp_reg`

Після завантаження з CSV вони зчитуються як текстові, тому їх необхідно перетворити у формат datetime.

In [4]:
train_transactions.info()
print()
train_users.info()

<class 'pandas.DataFrame'>
RangeIndex: 3135378 entries, 0 to 3135377
Data columns (total 13 columns):
 #   Column            Dtype  
---  ------            -----  
 0   id_user           int64  
 1   timestamp_tr      str    
 2   amount            float64
 3   status            str    
 4   transaction_type  str    
 5   error_group       str    
 6   currency          str    
 7   card_brand        str    
 8   card_type         str    
 9   card_country      str    
 10  card_holder       str    
 11  card_mask_hash    str    
 12  payment_country   str    
dtypes: float64(1), int64(1), str(11)
memory usage: 311.0 MB

<class 'pandas.DataFrame'>
RangeIndex: 395381 entries, 0 to 395380
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id_user        395381 non-null  int64
 1   timestamp_reg  395381 non-null  str  
 2   email          395063 non-null  str  
 3   gender         395381 non-null  str  
 4   reg_count

## 5. Аналіз пропущених значень

На цьому етапі перевіряємо, в яких колонках є пропуски.

Особливо важливо правильно інтерпретувати пропуски:
- велика кількість `NaN` у `error_group` є очікуваною, оскільки для успішних транзакцій помилка відсутня;
- пропуски у `card_country`, `payment_country`, `card_holder`, `card_mask_hash` можуть бути окремим сигналом ризику;
- у таблиці користувачів пропусків небагато, тому вони не створюють суттєвої проблеми.

У fraud-задачах не варто агресивно видаляти "дивні" або неповні записи, тому що саме вони можуть бути інформативними.

In [5]:
print(train_transactions.isnull().sum())
print()
print(train_users.isnull().sum())

id_user                   0
timestamp_tr              0
amount                    0
status                    0
transaction_type          0
error_group         1948264
currency                  0
card_brand            31941
card_type             33958
card_country          32996
card_holder          450178
card_mask_hash        20950
payment_country       68608
dtype: int64

id_user            0
timestamp_reg      0
email            318
gender             0
reg_country      126
traffic_type       0
is_fraud           0
dtype: int64


## 6. Базова підготовка даних

Наступний крок — перетворення часових колонок у формат datetime.

Це потрібно для:
- обчислення різниці часу між транзакціями;
- створення часових фіч (година, день, місяць, день тижня);
- обчислення часу від реєстрації до першої транзакції.

Параметр `format='mixed'` дозволяє коректно розібрати змішані формати часових міток, а `utc=True` приводить усі дати до єдиної часової зони.

In [6]:
train_transactions['timestamp_tr'] = pd.to_datetime(
    train_transactions['timestamp_tr'],
    format='mixed',
    utc=True
)

train_users['timestamp_reg'] = pd.to_datetime(
    train_users['timestamp_reg'],
    format='mixed',
    utc=True
)

test_transactions['timestamp_tr'] = pd.to_datetime(
    test_transactions['timestamp_tr'],
    format='mixed',
    utc=True
)

test_users['timestamp_reg'] = pd.to_datetime(
    test_users['timestamp_reg'],
    format='mixed',
    utc=True
)

## 7. Аналіз цільової змінної

Перевіримо баланс класів у train-частині. Це один із ключових етапів, оскільки від нього залежить вибір метрики та стратегія моделювання.

У цій задачі частка fraud-користувачів невелика, тому метрика accuracy не є показовою. Саме тому основною метрикою є **F1-score**, яка балансує precision і recall.

In [7]:
print(train_users['is_fraud'].value_counts())
print()
print(train_users['is_fraud'].value_counts(normalize=True))

is_fraud
0    380449
1     14932
Name: count, dtype: int64

is_fraud
0    0.962234
1    0.037766
Name: proportion, dtype: float64


## 8. Базовий EDA транзакцій

На цьому етапі дивимося на загальні поведінкові патерни в транзакціях:
- скільки транзакцій припадає на користувача;
- співвідношення успішних і неуспішних платежів;
- розподіл типів транзакцій;
- розподіл помилок;
- частку географічних mismatch між країною картки та країною платежу.

Це дозволяє сформулювати перші гіпотези для anti-fraud ознак.

In [8]:
print(train_transactions.groupby('id_user').size().describe())
print()
print(train_transactions['status'].value_counts())
print()
print(train_transactions['error_group'].value_counts())
print()
print(train_transactions['transaction_type'].value_counts())
print()
print((train_transactions['card_country'] != train_transactions['payment_country']).mean())

count    395381.000000
mean          7.930017
std          44.414024
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max        4762.000000
dtype: float64

status
success    1929295
fail       1206083
Name: count, dtype: int64

error_group
insufficient funds error    339521
fraud                       237785
3ds error                   179524
do not honor                107024
antifraud                    84066
limit exceeded error         66498
issuer decline               48238
card problem                 33660
invalid data                 23686
cvv error                    23065
decline (general)            16509
technical issue              11670
expired error                 7719
merchant problem              7223
decline (other)                817
user decline                    73
token error                     36
Name: count, dtype: int64

transaction_type
card_recurring    1685790
card_init          742520
google-pay         411525

## 9. Побудова базових user-level фіч

Оскільки цільова змінна `is_fraud` задана на рівні користувача, транзакції потрібно агрегувати до user-level.

На цьому етапі будуємо базові агрегати:
- кількість транзакцій;
- середня, максимальна, мінімальна сума;
- стандартне відхилення суми;
- частка failed транзакцій;
- кількість унікальних карт;
- кількість унікальних країн картки та платежу.

Ці ознаки описують загальну платіжну поведінку користувача.

In [9]:
user_features = train_transactions.groupby('id_user').agg(
    transaction_count=('id_user', 'count'),
    avg_amount=('amount', 'mean'),
    max_amount=('amount', 'max'),
    min_amount=('amount', 'min'),
    std_amount=('amount', 'std'),
    fail_ratio=('status', lambda x: (x == 'fail').mean()),
    unique_cards=('card_mask_hash', 'nunique'),
    unique_card_country=('card_country', 'nunique'),
    unique_payment_country=('payment_country', 'nunique')
).reset_index()

train_final = train_users.merge(user_features, on='id_user', how='left')

train_final.head()
print()
print(train_final.isnull().sum())


id_user                        0
timestamp_reg                  0
email                        318
gender                         0
reg_country                  126
traffic_type                   0
is_fraud                       0
transaction_count              0
avg_amount                     0
max_amount                     0
min_amount                     0
std_amount                176503
fail_ratio                     0
unique_cards                   0
unique_card_country            0
unique_payment_country         0
dtype: int64


## 10. Обробка стандартного відхилення суми

Після агрегації видно, що колонка `std_amount` має значну кількість пропусків. Це очікувано: стандартне відхилення не обчислюється для користувачів, у яких була лише одна транзакція.

Тому такі пропуски не є помилкою. Їх логічно замінити на 0, що відповідає відсутності варіативності в сумах.

In [10]:
train_final['std_amount'] = train_final['std_amount'].fillna(0)

## 11. Перший baseline-модельний підхід: RandomForest

Спочатку будуємо простий baseline на агрегованих фічах. Для цього:
- додаємо часові ознаки реєстрації;
- виділяємо домен email;
- кодуємо категоріальні змінні через one-hot encoding;
- ділимо train на train/validation;
- навчаємо RandomForest.

Цей baseline потрібен не як фінальне рішення, а як відправна точка, з якою можна порівнювати подальші покращення.

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

train_model_df = train_final.copy()

train_model_df['std_amount'] = train_model_df['std_amount'].fillna(0)

train_model_df['reg_hour'] = train_model_df['timestamp_reg'].dt.hour
train_model_df['reg_day'] = train_model_df['timestamp_reg'].dt.day
train_model_df['reg_weekday'] = train_model_df['timestamp_reg'].dt.weekday

train_model_df['email_domain'] = (
    train_model_df['email']
    .fillna('missing')
    .str.split('@')
    .str[-1]
)

top_domains = train_model_df['email_domain'].value_counts().head(20).index
train_model_df['email_domain'] = train_model_df['email_domain'].where(
    train_model_df['email_domain'].isin(top_domains),
    'other'
)

train_model_df = train_model_df.drop(columns=['timestamp_reg', 'email'])

X = train_model_df.drop(columns=['is_fraud', 'id_user'])
y = train_model_df['is_fraud']

cat_cols = ['gender', 'reg_country', 'traffic_type', 'email_domain']
X = pd.get_dummies(X, columns=cat_cols, drop_first=False)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    fold_f1 = f1_score(y_val, y_pred)
    f1_scores.append(fold_f1)

    print(f"Fold {fold} F1: {fold_f1:.6f}")

print("\nCV F1 scores:", [round(score, 6) for score in f1_scores])
print("Mean F1:", round(np.mean(f1_scores), 6))
print("Std F1:", round(np.std(f1_scores), 6))

Fold 1 F1: 0.309226
Fold 2 F1: 0.321827
Fold 3 F1: 0.310434
Fold 4 F1: 0.319757
Fold 5 F1: 0.317364

CV F1 scores: [0.309226, 0.321827, 0.310434, 0.319757, 0.317364]
Mean F1: 0.315722
Std F1: 0.005028


## 12. Додаткові anti-fraud фічі

Щоб покращити baseline, додаємо більш спеціалізовані anti-fraud ознаки:

### Географічні ознаки
- `geo_mismatch_ratio` — частка транзакцій, де країна картки не збігається з країною платежу;
- `missing_country_ratio` — частка транзакцій з відсутньою географічною інформацією.

### Ознаки помилок
- `cvv_error_ratio` — частка транзакцій з CVV-related помилками;
- `antifraud_ratio` — частка транзакцій, які потрапили під антифрод-обмеження.

### Часові ознаки
- `avg_time_between` — середній інтервал між транзакціями;
- `min_time_between` — мінімальний інтервал між транзакціями.

Ці фічі допомагають краще описати ризикову поведінку користувача.

In [12]:
# --- GEO FEATURES ---
train_transactions['geo_mismatch'] = (
    train_transactions['card_country'].notna() &
    train_transactions['payment_country'].notna() &
    (train_transactions['card_country'] != train_transactions['payment_country'])
).astype(int)

train_transactions['missing_country_info'] = (
    train_transactions['card_country'].isna() |
    train_transactions['payment_country'].isna()
).astype(int)

geo_features = train_transactions.groupby('id_user').agg(
    geo_mismatch_ratio=('geo_mismatch', 'mean'),
    missing_country_ratio=('missing_country_info', 'mean')
).reset_index()

# --- ERROR FEATURES ---
train_transactions['is_cvv_error'] = (train_transactions['error_group'] == 'cvv error').astype(int)
train_transactions['is_antifraud'] = (train_transactions['error_group'] == 'antifraud').astype(int)

error_features = train_transactions.groupby('id_user').agg(
    cvv_error_ratio=('is_cvv_error', 'mean'),
    antifraud_ratio=('is_antifraud', 'mean')
).reset_index()

# --- TIME FEATURES ---
train_transactions = train_transactions.sort_values(['id_user', 'timestamp_tr'])

train_transactions['time_diff'] = (
    train_transactions.groupby('id_user')['timestamp_tr']
    .diff()
    .dt.total_seconds()
)

time_features = train_transactions.groupby('id_user').agg(
    avg_time_between=('time_diff', 'mean'),
    min_time_between=('time_diff', 'min')
).reset_index()

# --- MERGE ---
train_final = train_final.merge(geo_features, on='id_user', how='left')
train_final = train_final.merge(error_features, on='id_user', how='left')
train_final = train_final.merge(time_features, on='id_user', how='left')

# --- FILLNA AFTER MERGE ---
train_final['geo_mismatch_ratio'] = train_final['geo_mismatch_ratio'].fillna(0)
train_final['missing_country_ratio'] = train_final['missing_country_ratio'].fillna(0)
train_final['cvv_error_ratio'] = train_final['cvv_error_ratio'].fillna(0)
train_final['antifraud_ratio'] = train_final['antifraud_ratio'].fillna(0)
train_final['avg_time_between'] = train_final['avg_time_between'].fillna(0)
train_final['min_time_between'] = train_final['min_time_between'].fillna(0)

## 13. Розширення user-level фіч

Далі додаємо ще кілька поведінкових фіч, які виявилися корисними:

- `seconds_to_first_transaction` — час від реєстрації до першої транзакції;
- `success_ratio` — частка успішних транзакцій;
- `max_tx_per_day` — максимальна кількість транзакцій за день;
- `avg_tx_per_day` — середня кількість транзакцій за день;
- `unique_transaction_types` — кількість різних типів транзакцій;
- `card_init_ratio` — частка транзакцій типу `card_init`.

Ці фічі описують швидкість активації акаунта, інтенсивність використання та різноманітність поведінки.

In [13]:
# A. Час від реєстрації до першої транзакції
first_tr = (
    train_transactions.groupby('id_user')['timestamp_tr']
    .min()
    .reset_index(name='first_transaction_time')
)

train_final = train_final.merge(first_tr, on='id_user', how='left')

train_final['seconds_to_first_transaction'] = (
    train_final['first_transaction_time'] - train_final['timestamp_reg']
).dt.total_seconds()

train_final['seconds_to_first_transaction'] = train_final['seconds_to_first_transaction'].fillna(0)
train_final = train_final.drop(columns=['first_transaction_time'])

# B. Частка success
success_features = train_transactions.groupby('id_user').agg(
    success_ratio=('status', lambda x: (x == 'success').mean())
).reset_index()

train_final = train_final.merge(success_features, on='id_user', how='left')
train_final['success_ratio'] = train_final['success_ratio'].fillna(0)

# C. Найбільша кількість транзакцій за день
train_transactions['tr_date'] = train_transactions['timestamp_tr'].dt.date

daily_counts = (
    train_transactions.groupby(['id_user', 'tr_date'])
    .size()
    .reset_index(name='daily_tx_count')
)

daily_features = daily_counts.groupby('id_user').agg(
    max_tx_per_day=('daily_tx_count', 'max'),
    avg_tx_per_day=('daily_tx_count', 'mean')
).reset_index()

train_final = train_final.merge(daily_features, on='id_user', how='left')
train_final['max_tx_per_day'] = train_final['max_tx_per_day'].fillna(0)
train_final['avg_tx_per_day'] = train_final['avg_tx_per_day'].fillna(0)

# D. Кількість унікальних transaction_type
type_features = train_transactions.groupby('id_user').agg(
    unique_transaction_types=('transaction_type', 'nunique')
).reset_index()

train_final = train_final.merge(type_features, on='id_user', how='left')
train_final['unique_transaction_types'] = train_final['unique_transaction_types'].fillna(0)

# E. Частка card_init
train_transactions['is_card_init'] = (
    train_transactions['transaction_type'] == 'card_init'
).astype(int)

card_init_features = train_transactions.groupby('id_user').agg(
    card_init_ratio=('is_card_init', 'mean')
).reset_index()

train_final = train_final.merge(card_init_features, on='id_user', how='left')
train_final['card_init_ratio'] = train_final['card_init_ratio'].fillna(0)

# там є один момент, який не дуже принциповий, але можна почистити

## 14. Покращений RandomForest і підбір threshold

Після додавання нових ознак повторно навчаємо RandomForest і підбираємо threshold не за замовчуванням (0.5), а за критерієм максимального F1-score.

Це важливо, тому що в незбалансованих задачах стандартний поріг 0.5 часто не є оптимальним.

In [14]:
train_model_df = train_final.copy()

fill_zero_cols = [
    'std_amount',
    'geo_mismatch_ratio',
    'missing_country_ratio',
    'cvv_error_ratio',
    'antifraud_ratio',
    'avg_time_between',
    'min_time_between',
    'seconds_to_first_transaction',
    'success_ratio',
    'max_tx_per_day',
    'avg_tx_per_day',
    'unique_transaction_types',
    'card_init_ratio'
]

for col in fill_zero_cols:
    if col in train_model_df.columns:
        train_model_df[col] = train_model_df[col].fillna(0)

train_model_df['reg_hour'] = train_model_df['timestamp_reg'].dt.hour
train_model_df['reg_day'] = train_model_df['timestamp_reg'].dt.day
train_model_df['reg_weekday'] = train_model_df['timestamp_reg'].dt.weekday
train_model_df['reg_month'] = train_model_df['timestamp_reg'].dt.month

train_model_df['email_domain'] = (
    train_model_df['email']
    .fillna('missing')
    .str.split('@')
    .str[-1]
)

top_domains = train_model_df['email_domain'].value_counts().head(20).index
train_model_df['email_domain'] = train_model_df['email_domain'].where(
    train_model_df['email_domain'].isin(top_domains),
    'other'
)

train_model_df = train_model_df.drop(columns=['timestamp_reg', 'email'])

X = train_model_df.drop(columns=['is_fraud', 'id_user'])
y = train_model_df['is_fraud']

cat_cols = ['gender', 'reg_country', 'traffic_type', 'email_domain']
X = pd.get_dummies(X, columns=cat_cols, drop_first=False)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_proba = model.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.02)

best_f1 = 0
best_t = 0.5

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    f1 = f1_score(y_val, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best F1-score:", best_f1)

best_pred = (y_proba >= best_t).astype(int)

print()
print(classification_report(y_val, best_pred))

Best threshold: 0.30000000000000004
Best F1-score: 0.5520632133450395

              precision    recall  f1-score   support

           0       0.98      0.99      0.98     76091
           1       0.58      0.53      0.55      2986

    accuracy                           0.97     79077
   macro avg       0.78      0.76      0.77     79077
weighted avg       0.97      0.97      0.97     79077



## 15. Перехід на CatBoost

Оскільки дані містять багато категоріальних ознак, наступним кроком є використання `CatBoostClassifier`, який добре працює з табличними даними та природно підтримує categorical features.

На відміну від RandomForest з one-hot encoding, CatBoost дозволяє:
- зберігати категоріальні ознаки у сирому вигляді;
- ефективно працювати з категоріями без роздування кількості колонок;
- краще моделювати нелінійні залежності в tabular data.

Після навчання моделі знову виконується threshold tuning.

In [15]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import numpy as np
import pandas as pd

train_model_df = train_final.copy()

fill_zero_cols = [
    'std_amount',
    'geo_mismatch_ratio',
    'missing_country_ratio',
    'cvv_error_ratio',
    'antifraud_ratio',
    'avg_time_between',
    'min_time_between',
    'seconds_to_first_transaction',
    'success_ratio',
    'max_tx_per_day',
    'avg_tx_per_day',
    'unique_transaction_types',
    'card_init_ratio'
]

for col in fill_zero_cols:
    if col in train_model_df.columns:
        train_model_df[col] = train_model_df[col].fillna(0)

train_model_df['reg_hour'] = train_model_df['timestamp_reg'].dt.hour
train_model_df['reg_day'] = train_model_df['timestamp_reg'].dt.day
train_model_df['reg_weekday'] = train_model_df['timestamp_reg'].dt.weekday
train_model_df['reg_month'] = train_model_df['timestamp_reg'].dt.month

train_model_df['email_domain'] = (
    train_model_df['email']
    .fillna('missing')
    .str.split('@')
    .str[-1]
)

top_domains = train_model_df['email_domain'].value_counts().head(20).index
train_model_df['email_domain'] = train_model_df['email_domain'].where(
    train_model_df['email_domain'].isin(top_domains),
    'other'
)

train_model_df = train_model_df.drop(columns=['timestamp_reg', 'email'])

cat_features = ['gender', 'reg_country', 'traffic_type', 'email_domain']

for col in cat_features:
    train_model_df[col] = train_model_df[col].fillna('missing').astype(str)

X = train_model_df.drop(columns=['is_fraud', 'id_user'])
y = train_model_df['is_fraud']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

thresholds = np.arange(0.10, 0.91, 0.02)

fold_results = []
best_fold_report = None

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        eval_metric='F1',
        verbose=100
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val)
    )

    y_proba = model.predict_proba(X_val)[:, 1]

    best_f1 = 0
    best_t = 0.5
    best_pred = None

    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_val, y_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t
            best_pred = y_pred

    fold_results.append({
        'fold': fold,
        'best_threshold': best_t,
        'best_f1': best_f1
    })

    print(f"\nFold {fold}")
    print("Best threshold:", best_t)
    print("Best F1-score:", round(best_f1, 6))
    print(classification_report(y_val, best_pred))

results_df = pd.DataFrame(fold_results)

print("\n" + "=" * 60)
print("CV RESULTS")
print("=" * 60)
print(results_df)

print("\nMean F1:", round(results_df['best_f1'].mean(), 6))
print("Std F1:", round(results_df['best_f1'].std(), 6))
print("Mean threshold:", round(results_df['best_threshold'].mean(), 6))

0:	learn: 0.3354466	test: 0.3337497	best: 0.3337497 (0)	total: 171ms	remaining: 1m 25s
100:	learn: 0.4699009	test: 0.4692202	best: 0.4692202 (100)	total: 15.3s	remaining: 1m
200:	learn: 0.5052399	test: 0.4902519	best: 0.4911281 (199)	total: 28.4s	remaining: 42.3s
300:	learn: 0.5256424	test: 0.5013010	best: 0.5021683 (282)	total: 42.9s	remaining: 28.4s
400:	learn: 0.5393613	test: 0.5058366	best: 0.5069085 (397)	total: 59s	remaining: 14.6s
499:	learn: 0.5497044	test: 0.5096691	best: 0.5097786 (498)	total: 1m 14s	remaining: 0us

bestTest = 0.5097786374
bestIteration = 498

Shrink model to first 499 iterations.

Fold 1
Best threshold: 0.24000000000000002
Best F1-score: 0.572592
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     76090
           1       0.54      0.61      0.57      2987

    accuracy                           0.97     79077
   macro avg       0.76      0.80      0.78     79077
weighted avg       0.97      0.97      0.97  

## 16. Підбір найкращої конфігурації CatBoost

Після перевірки базової моделі CatBoost за допомогою StratifiedKFold було проведено порівняння кількох конфігурацій гіперпараметрів.

Для кожної конфігурації:
- виконувалось навчання на 5 фолдах;
- у кожному фолді підбирався оптимальний threshold;
- обчислювався F1-score;
- далі розраховувалися середнє значення F1-score та стандартне відхилення.

Такий підхід дозволяє обрати не просто модель, яка випадково добре спрацювала на одному розбитті, а конфігурацію, що стабільно показує високий результат на різних підмножинах даних.

In [16]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# -------------------------------------------------
# Підготовка даних (аналогічно пункту 15)
# -------------------------------------------------

train_model_df = train_final.copy()

fill_zero_cols = [
    'std_amount',
    'geo_mismatch_ratio',
    'missing_country_ratio',
    'cvv_error_ratio',
    'antifraud_ratio',
    'avg_time_between',
    'min_time_between',
    'seconds_to_first_transaction',
    'success_ratio',
    'max_tx_per_day',
    'avg_tx_per_day',
    'unique_transaction_types',
    'card_init_ratio'
]

for col in fill_zero_cols:
    if col in train_model_df.columns:
        train_model_df[col] = train_model_df[col].fillna(0)

train_model_df['reg_hour'] = train_model_df['timestamp_reg'].dt.hour
train_model_df['reg_day'] = train_model_df['timestamp_reg'].dt.day
train_model_df['reg_weekday'] = train_model_df['timestamp_reg'].dt.weekday
train_model_df['reg_month'] = train_model_df['timestamp_reg'].dt.month

train_model_df['email_domain'] = (
    train_model_df['email']
    .fillna('missing')
    .str.split('@')
    .str[-1]
)

top_domains = train_model_df['email_domain'].value_counts().head(20).index
train_model_df['email_domain'] = train_model_df['email_domain'].where(
    train_model_df['email_domain'].isin(top_domains),
    'other'
)

train_model_df = train_model_df.drop(columns=['timestamp_reg', 'email'])

cat_features = ['gender', 'reg_country', 'traffic_type', 'email_domain']

for col in cat_features:
    train_model_df[col] = train_model_df[col].fillna('missing').astype(str)

X = train_model_df.drop(columns=['is_fraud', 'id_user'])
y = train_model_df['is_fraud']

# -------------------------------------------------
# Конфігурації моделей
# -------------------------------------------------

configs = [
    {
        'name': 'try_1',
        'iterations': 500,
        'depth': 8,
        'learning_rate': 0.05,
        'loss_function': 'Logloss',
        'eval_metric': 'F1',
        'verbose': 100
    },
    {
        'name': 'try_2',
        'iterations': 800,
        'depth': 6,
        'learning_rate': 0.03,
        'loss_function': 'Logloss',
        'eval_metric': 'F1',
        'verbose': 100
    },
    {
        'name': 'try_3',
        'iterations': 800,
        'depth': 8,
        'learning_rate': 0.03,
        'loss_function': 'Logloss',
        'eval_metric': 'F1',
        'verbose': 100
    },
    {
        'name': 'try_4',
        'iterations': 800,
        'depth': 7,
        'learning_rate': 0.03,
        'l2_leaf_reg': 5,
        'loss_function': 'Logloss',
        'eval_metric': 'F1',
        'verbose': 100
    }
]

thresholds = np.arange(0.10, 0.91, 0.02)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
best_model_name = None
best_overall_mean_f1 = -1
best_overall_threshold = None
best_config = None

# -------------------------------------------------
# CV-порівняння конфігурацій
# -------------------------------------------------

for cfg in configs:
    print("=" * 80)
    print(f"Training config: {cfg['name']}")
    print(cfg)
    print("=" * 80)

    fold_f1_scores = []
    fold_thresholds = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostClassifier(
            iterations=cfg['iterations'],
            depth=cfg['depth'],
            learning_rate=cfg['learning_rate'],
            loss_function=cfg['loss_function'],
            eval_metric=cfg['eval_metric'],
            verbose=cfg['verbose'],
            **({'l2_leaf_reg': cfg['l2_leaf_reg']} if 'l2_leaf_reg' in cfg else {})
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50
        )

        y_proba = model.predict_proba(X_val)[:, 1]

        best_f1 = 0
        best_t = 0.5

        for t in thresholds:
            y_pred = (y_proba >= t).astype(int)
            f1 = f1_score(y_val, y_pred)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        fold_f1_scores.append(best_f1)
        fold_thresholds.append(best_t)

        print(f"\n{cfg['name']} | Fold {fold}")
        print("Best threshold:", round(best_t, 4))
        print("Best F1-score:", round(best_f1, 6))

    mean_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)
    mean_threshold = np.mean(fold_thresholds)

    print("\n" + "-" * 80)
    print(f"{cfg['name']} SUMMARY")
    print("-" * 80)
    print("Fold F1 scores:", [round(x, 6) for x in fold_f1_scores])
    print("Fold thresholds:", [round(x, 4) for x in fold_thresholds])
    print("Mean F1:", round(mean_f1, 6))
    print("Std F1:", round(std_f1, 6))
    print("Mean threshold:", round(mean_threshold, 6))
    print()

    results.append({
        'model_name': cfg['name'],
        'iterations': cfg['iterations'],
        'depth': cfg['depth'],
        'learning_rate': cfg['learning_rate'],
        'l2_leaf_reg': cfg.get('l2_leaf_reg', None),
        'mean_threshold': mean_threshold,
        'mean_f1': mean_f1,
        'std_f1': std_f1
    })

    if mean_f1 > best_overall_mean_f1:
        best_overall_mean_f1 = mean_f1
        best_overall_threshold = mean_threshold
        best_model_name = cfg['name']
        best_config = cfg

# -------------------------------------------------
# Фінальна таблиця порівняння
# -------------------------------------------------

results_df = (
    pd.DataFrame(results)
    .sort_values(['mean_f1', 'std_f1'], ascending=[False, True])
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("FINAL CV COMPARISON TABLE")
print("=" * 80)
print(results_df)

print("\n" + "=" * 80)
print("BEST CONFIG OVERALL")
print("=" * 80)
print("Model:", best_model_name)
print("Best mean CV F1-score:", round(best_overall_mean_f1, 6))
print("Recommended threshold:", round(best_overall_threshold, 6))
print("Best config:", best_config)

Training config: try_1
{'name': 'try_1', 'iterations': 500, 'depth': 8, 'learning_rate': 0.05, 'loss_function': 'Logloss', 'eval_metric': 'F1', 'verbose': 100}
0:	learn: 0.3353647	test: 0.3333333	best: 0.3333333 (0)	total: 145ms	remaining: 1m 12s
100:	learn: 0.4951188	test: 0.4798560	best: 0.4798560 (100)	total: 19.6s	remaining: 1m 17s
200:	learn: 0.5384153	test: 0.5023820	best: 0.5024908 (197)	total: 36.8s	remaining: 54.8s
300:	learn: 0.5657223	test: 0.5087078	best: 0.5100988 (291)	total: 55.4s	remaining: 36.6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5100988397
bestIteration = 291

Shrink model to first 292 iterations.

try_1 | Fold 1
Best threshold: 0.28
Best F1-score: 0.573086
0:	learn: 0.3642436	test: 0.3573320	best: 0.3573320 (0)	total: 128ms	remaining: 1m 3s
100:	learn: 0.4871678	test: 0.4855363	best: 0.4873051 (98)	total: 19.3s	remaining: 1m 16s
200:	learn: 0.5358003	test: 0.5122004	best: 0.5122004 (198)	total: 36.7s	remaining: 54.5s
Stopped by overfi

300:	learn: 0.5309055	test: 0.5002182	best: 0.5004359 (299)	total: 51.6s	remaining: 1m 25s
400:	learn: 0.5493487	test: 0.5059511	best: 0.5059511 (400)	total: 1m 8s	remaining: 1m 8s
500:	learn: 0.5662160	test: 0.5089420	best: 0.5101249 (486)	total: 1m 25s	remaining: 51.2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5108672262
bestIteration = 510

Shrink model to first 511 iterations.

try_3 | Fold 1
Best threshold: 0.26
Best F1-score: 0.571382
0:	learn: 0.3642436	test: 0.3573320	best: 0.3573320 (0)	total: 137ms	remaining: 1m 49s
100:	learn: 0.4553161	test: 0.4521699	best: 0.4523199 (98)	total: 16.8s	remaining: 1m 56s
200:	learn: 0.5022954	test: 0.4919051	best: 0.4921303 (198)	total: 34s	remaining: 1m 41s
300:	learn: 0.5294182	test: 0.5052129	best: 0.5059718 (294)	total: 50.4s	remaining: 1m 23s
400:	learn: 0.5494446	test: 0.5150732	best: 0.5153929 (399)	total: 1m 8s	remaining: 1m 7s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5153928956
bes

## 17. Фінальний вибір моделі

Після порівняння кількох конфігурацій CatBoost за допомогою StratifiedKFold було обрано найкращу фінальну модель.

Оцінювання виконувалося за схемою 5-fold cross-validation, що дозволило:
- порівняти конфігурації на різних підмножинах даних;
- оцінити не лише середній F1-score, а й стабільність моделі;
- зменшити вплив випадкового train/validation split на фінальний вибір.

Найкращою конфігурацією виявилась модель CatBoost (try_3), для якої отримано:

- Mean F1-score ≈ 0.5737  
- Std F1-score ≈ 0.0095  
- Mean threshold ≈ 0.292  

Невелике стандартне відхилення свідчить про стабільну роботу моделі на різних фолдах.

Фінальна конфігурація моделі:

- Model: CatBoost (try_3)
- iterations: 800
- depth: 8
- learning_rate: 0.03
- Threshold: 0.29

Ця конфігурація показала найкращий середній результат серед усіх протестованих варіантів, тому саме вона була обрана для фінального навчання на всьому train-наборі та подальшого прогнозування на test-даних.

In [17]:
# Фінальна обрана конфігурація (на основі CV-порівняння конфігурацій)

best_model_name = "CatBoost (try_3)"

best_threshold = 0.29   # округлено з mean threshold = 0.292

mean_f1_score = 0.573654
std_f1_score = 0.009450
mean_threshold = 0.292

print("Best model:", best_model_name)
print("Best threshold:", best_threshold)
print("Mean CV F1-score:", round(mean_f1_score, 6))
print("Std CV F1-score:", round(std_f1_score, 6))
print("Mean CV threshold:", round(mean_threshold, 6))

Best model: CatBoost (try_3)
Best threshold: 0.29
Mean CV F1-score: 0.573654
Std CV F1-score: 0.00945
Mean CV threshold: 0.292


## 18. Висновок

У межах роботи було побудовано anti-fraud pipeline для виявлення шахрайських користувачів на основі транзакційної історії та атрибутів акаунта.

Основні результати:

- виконано базову підготовку та аналіз даних;
- побудовано user-level агрегати з транзакцій;
- створено спеціалізовані anti-fraud фічі (географічні, помилкові, часові та поведінкові ознаки);
- протестовано моделі RandomForest та CatBoost;
- виконано підбір оптимального порогу класифікації;
- впроваджено StratifiedKFold для більш надійної оцінки якості моделі;
- проведено порівняння кількох конфігурацій CatBoost за допомогою крос-валідації.

Найкращий результат показала модель CatBoost (try_3), яка досягла:

- Mean F1-score ≈ 0.5737  
- Std ≈ 0.0095  

Це свідчить про стабільну роботу моделі на різних підмножинах даних.

Отримана модель є фінальним кандидатом для навчання на всьому train-наборі та подальшого застосування до test-даних для формування submission-файлу.

# 19. Фінальне навчання моделі та побудова submission    
У цьому розділі виконується повний pipeline:

побудова test-фіч (аналогічно train);
навчання фінальної моделі на всьому train;
прогнозування на test;
формування submission.

# 19.1 Побудова test user-level фіч

In [28]:
# ================================
# TEST: GEO FEATURES
# ================================

test_transactions['geo_mismatch'] = (
    test_transactions['card_country'].notna() &
    test_transactions['payment_country'].notna() &
    (test_transactions['card_country'] != test_transactions['payment_country'])
).astype(int)

test_transactions['missing_country_info'] = (
    test_transactions['card_country'].isna() |
    test_transactions['payment_country'].isna()
).astype(int)

geo_features_test = test_transactions.groupby('id_user').agg(
    geo_mismatch_ratio=('geo_mismatch', 'mean'),
    missing_country_ratio=('missing_country_info', 'mean')
).reset_index()


# ================================
# TEST: ERROR FEATURES
# ================================

test_transactions['is_cvv_error'] = (test_transactions['error_group'] == 'cvv error').astype(int)
test_transactions['is_antifraud'] = (test_transactions['error_group'] == 'antifraud').astype(int)

error_features_test = test_transactions.groupby('id_user').agg(
    cvv_error_ratio=('is_cvv_error', 'mean'),
    antifraud_ratio=('is_antifraud', 'mean')
).reset_index()


# ================================
# TEST: TIME FEATURES
# ================================

test_transactions = test_transactions.sort_values(['id_user', 'timestamp_tr'])

test_transactions['time_diff'] = (
    test_transactions.groupby('id_user')['timestamp_tr']
    .diff()
    .dt.total_seconds()
)

time_features_test = test_transactions.groupby('id_user').agg(
    avg_time_between=('time_diff', 'mean'),
    min_time_between=('time_diff', 'min')
).reset_index()


# ================================
# TEST: BASE USER FEATURES
# ================================

user_features_test = test_transactions.groupby('id_user').agg(
    transaction_count=('id_user', 'count'),
    avg_amount=('amount', 'mean'),
    max_amount=('amount', 'max'),
    min_amount=('amount', 'min'),
    std_amount=('amount', 'std'),
    fail_ratio=('status', lambda x: (x == 'fail').mean()),
    unique_cards=('card_mask_hash', 'nunique'),
    unique_card_country=('card_country', 'nunique'),
    unique_payment_country=('payment_country', 'nunique')
).reset_index()


# ================================
# TEST: FIRST TRANSACTION TIME
# ================================

first_tr_test = (
    test_transactions.groupby('id_user')['timestamp_tr']
    .min()
    .reset_index(name='first_transaction_time')
)

# ================================
# TEST: SUCCESS RATIO
# ================================

success_features_test = test_transactions.groupby('id_user').agg(
    success_ratio=('status', lambda x: (x == 'success').mean())
).reset_index()


# ================================
# TEST: DAILY FEATURES
# ================================

test_transactions['tr_date'] = test_transactions['timestamp_tr'].dt.date

daily_counts_test = (
    test_transactions.groupby(['id_user', 'tr_date'])
    .size()
    .reset_index(name='daily_tx_count')
)

daily_features_test = daily_counts_test.groupby('id_user').agg(
    max_tx_per_day=('daily_tx_count', 'max'),
    avg_tx_per_day=('daily_tx_count', 'mean')
).reset_index()


# ================================
# TEST: TYPE FEATURES
# ================================

type_features_test = test_transactions.groupby('id_user').agg(
    unique_transaction_types=('transaction_type', 'nunique')
).reset_index()


# ================================
# TEST: CARD INIT RATIO
# ================================

test_transactions['is_card_init'] = (
    test_transactions['transaction_type'] == 'card_init'
).astype(int)

card_init_features_test = test_transactions.groupby('id_user').agg(
    card_init_ratio=('is_card_init', 'mean')
).reset_index()

# 19.2 Merge test-фіч

In [29]:
test_final = test_users.copy()

test_final = test_final.merge(user_features_test, on='id_user', how='left')
test_final = test_final.merge(geo_features_test, on='id_user', how='left')
test_final = test_final.merge(error_features_test, on='id_user', how='left')
test_final = test_final.merge(time_features_test, on='id_user', how='left')
test_final = test_final.merge(first_tr_test, on='id_user', how='left')
test_final = test_final.merge(success_features_test, on='id_user', how='left')
test_final = test_final.merge(daily_features_test, on='id_user', how='left')
test_final = test_final.merge(type_features_test, on='id_user', how='left')
test_final = test_final.merge(card_init_features_test, on='id_user', how='left')

# 19.3 Додавання похідних фіч

In [30]:
test_final['seconds_to_first_transaction'] = (
    test_final['first_transaction_time'] - test_final['timestamp_reg']
).dt.total_seconds()

test_final['seconds_to_first_transaction'] = test_final['seconds_to_first_transaction'].fillna(0)

test_final = test_final.drop(columns=['first_transaction_time'])

# 19.4 Обробка пропусків

In [31]:
fill_zero_cols = [
    'std_amount',
    'geo_mismatch_ratio',
    'missing_country_ratio',
    'cvv_error_ratio',
    'antifraud_ratio',
    'avg_time_between',
    'min_time_between',
    'seconds_to_first_transaction',
    'success_ratio',
    'max_tx_per_day',
    'avg_tx_per_day',
    'unique_transaction_types',
    'card_init_ratio'
]

for col in fill_zero_cols:
    if col in test_final.columns:
        test_final[col] = test_final[col].fillna(0)

# 19.5 Додаткові фічі користувача

In [32]:
test_final['reg_hour'] = test_final['timestamp_reg'].dt.hour
test_final['reg_day'] = test_final['timestamp_reg'].dt.day
test_final['reg_weekday'] = test_final['timestamp_reg'].dt.weekday
test_final['reg_month'] = test_final['timestamp_reg'].dt.month

test_final['email_domain'] = (
    test_final['email']
    .fillna('missing')
    .str.split('@')
    .str[-1]
)

# використовуємо top_domains, пораховані на train
test_final['email_domain'] = test_final['email_domain'].where(
    test_final['email_domain'].isin(top_domains),
    'other'
)

test_final = test_final.drop(columns=['timestamp_reg', 'email'])

# 19.6 Фінальне навчання моделі

In [33]:
from catboost import CatBoostClassifier

train_model_df = train_final.copy()

fill_zero_cols = [
    'std_amount',
    'geo_mismatch_ratio',
    'missing_country_ratio',
    'cvv_error_ratio',
    'antifraud_ratio',
    'avg_time_between',
    'min_time_between',
    'seconds_to_first_transaction',
    'success_ratio',
    'max_tx_per_day',
    'avg_tx_per_day',
    'unique_transaction_types',
    'card_init_ratio'
]

for col in fill_zero_cols:
    if col in train_model_df.columns:
        train_model_df[col] = train_model_df[col].fillna(0)

train_model_df['reg_hour'] = train_model_df['timestamp_reg'].dt.hour
train_model_df['reg_day'] = train_model_df['timestamp_reg'].dt.day
train_model_df['reg_weekday'] = train_model_df['timestamp_reg'].dt.weekday
train_model_df['reg_month'] = train_model_df['timestamp_reg'].dt.month

train_model_df['email_domain'] = (
    train_model_df['email']
    .fillna('missing')
    .str.split('@')
    .str[-1]
)

top_domains = train_model_df['email_domain'].value_counts().head(20).index

train_model_df['email_domain'] = train_model_df['email_domain'].where(
    train_model_df['email_domain'].isin(top_domains),
    'other'
)

cat_features = ['gender', 'reg_country', 'traffic_type', 'email_domain']

for col in cat_features:
    train_model_df[col] = train_model_df[col].fillna('missing').astype(str)

X_train = train_model_df.drop(columns=['is_fraud', 'id_user', 'timestamp_reg', 'email'])
y_train = train_model_df['is_fraud']

final_model = CatBoostClassifier(
    iterations=800,
    depth=8,
    learning_rate=0.03,
    loss_function='Logloss',
    eval_metric='F1',
    verbose=100
)

final_model.fit(X_train, y_train, cat_features=cat_features)

0:	learn: 0.3673665	total: 146ms	remaining: 1m 56s
100:	learn: 0.4556867	total: 18.9s	remaining: 2m 10s
200:	learn: 0.5008079	total: 49.8s	remaining: 2m 28s
300:	learn: 0.5273684	total: 1m 17s	remaining: 2m 8s
400:	learn: 0.5423936	total: 1m 47s	remaining: 1m 46s
500:	learn: 0.5559033	total: 2m 17s	remaining: 1m 22s
600:	learn: 0.5680083	total: 2m 46s	remaining: 55.2s
700:	learn: 0.5782298	total: 3m 13s	remaining: 27.3s
799:	learn: 0.5861965	total: 3m 45s	remaining: 0us


CatBoostClassifier(depth=8, eval_metric='F1', iterations=800, learning_rate=0.03, loss_function='Logloss', verbose=100)

# 19.7 Прогнозування на test

In [36]:
for col in cat_features:
    test_final[col] = test_final[col].fillna('missing').astype(str)

X_test = test_final.drop(columns=['id_user'])

y_test_proba = final_model.predict_proba(X_test)[:, 1]

threshold = 0.29

y_test_pred = (y_test_proba >= threshold).astype(int)

# sanity check

In [37]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Train/Test same columns:", list(X_train.columns) == list(X_test.columns))
print("NaN in X_train:", X_train.isna().sum().sum())
print("NaN in X_test:", X_test.isna().sum().sum())

X_train shape: (395381, 29)
X_test shape: (169449, 29)
Train/Test same columns: False
NaN in X_train: 0
NaN in X_test: 0


In [38]:
print(submission.head())
print(submission.shape)
print(submission['is_fraud'].value_counts())

    id_user  is_fraud
0  16318030         0
1  26996833         0
2  24252468         1
3  15990825         0
4  15349154         0
(169449, 2)
is_fraud
0    163423
1      6026
Name: count, dtype: int64


# 19.8 Формування submission

In [39]:
submission = pd.DataFrame({
    'id_user': test_final['id_user'],
    'is_fraud': y_test_pred
})

submission.to_csv('submission.csv', index=False)

print("Submission saved!")

Submission saved!
